# 03 · Peer segments (KMeans) and per-segment anomaly thresholds

A EUR 50,000 transfer is routine for a corporate account and a strong signal on a student's.
KMeans groups transactions into peer segments; the Isolation Forest's threshold is then set
**within each segment**, so each one flags its own most unusual 0.1%.

Runs after `02`: it reads `anomaly_isoforest.onnx`.

In [ ]:
import json, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))    # ml/, where sentinel_features.py lives
import sentinel_features as sf

sf.MODELS.mkdir(exist_ok=True)
N = len(sf.FEATURES)
rows = sf.load_rows()        # first run builds the features (~2 min), later runs read the cache
train, test, cutoff = sf.time_split(rows)
print(f"{len(rows):,} rows; train up to {cutoff}, {len(train):,} rows, {train.label.mean():.4%} laundering;"
      f" test {len(test):,} rows, {test.label.mean():.4%} laundering")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

fit_rows = sf.matrix(train.sample(300_000, random_state=42))
probe = fit_rows[:20_000]

scores = {}
for k in range(4, 9):
    candidate = make_pipeline(StandardScaler(), KMeans(n_clusters=k, n_init=5, random_state=42))
    labels = candidate.fit_predict(fit_rows)
    scores[k] = silhouette_score(candidate[0].transform(probe), labels[:20_000])
K = max(scores, key=scores.get)
print({k: round(s, 3) for k, s in scores.items()}, "-> k =", K)

segmenter = make_pipeline(StandardScaler(), KMeans(n_clusters=K, n_init=10, random_state=42))
segmenter.fit(fit_rows)

## What the segments are

Mean feature values per segment, and the laundering rate in each (labels used only to
describe, never to fit).

In [ ]:
described = train.sample(1_000_000, random_state=3)
described = described.assign(segment=segmenter.predict(sf.matrix(described)))
summary = described.groupby("segment")[sf.FEATURES].mean()
summary.insert(0, "share", described.segment.value_counts(normalize=True).sort_index())
summary.insert(1, "laundering_rate", described.groupby("segment").label.mean())
summary.round(3).T

## Per-segment thresholds

For each segment, the Isolation Forest score below which only 0.1% of that segment's
training transactions fall.

In [ ]:
import onnxruntime as ort

QUANTILE = 0.001
forest = ort.InferenceSession(str(sf.MODELS / "anomaly_isoforest.onnx"))

def forest_scores(frame):
    outputs = dict(zip([o.name for o in forest.get_outputs()],
                       forest.run(None, {"features": sf.matrix(frame)})))
    return outputs["scores"].ravel()

described["forest"] = forest_scores(described)
thresholds = described.groupby("segment").forest.quantile(QUANTILE)
global_threshold = described.forest.quantile(QUANTILE)
thresholds.round(4).to_dict(), round(global_threshold, 4)

## Does judging within the segment help?

Flag the test period both ways — per-segment thresholds, and one threshold over all training
rows, each at the most unusual 0.1% — and compare how many flags are real laundering.

In [ ]:
checked = test.sample(1_000_000, random_state=5)
checked = checked.assign(segment=segmenter.predict(sf.matrix(checked)), forest=forest_scores(checked))
per_segment = checked.forest < checked.segment.map(thresholds)
overall = checked.forest < global_threshold
pd.DataFrame({
    "flags": [per_segment.sum(), overall.sum()],
    "laundering among flags": [checked.label[per_segment].mean(), checked.label[overall].mean()],
    "laundering caught": [checked.label[per_segment].sum(), checked.label[overall].sum()],
}, index=["per-segment threshold", "one global threshold"]).round(4)

## Export

In [ ]:
import onnx
import onnxruntime as ort

def attach(model_proto, **metadata):
    """Metadata travels inside the .onnx file, so it cannot drift from the model it describes."""
    for key, value in metadata.items():
        entry = model_proto.metadata_props.add()
        entry.key, entry.value = key, json.dumps(value)
    return model_proto

def save(model_proto, name):
    path = sf.MODELS / f"{name}.onnx"
    onnx.save(model_proto, path)
    (sf.MODELS / f"{name}.features.json").write_text(json.dumps(sf.FEATURES, indent=2) + "\n")
    print(f"wrote {path.name} ({path.stat().st_size / 1024:.0f} KB) and {name}.features.json")
    return ort.InferenceSession(str(path))

# A handful of real rows, fraud included, that scoring-service's parity test replays through
# ONNX Runtime in Java and compares with what Python got here.
parity_rows = pd.concat([test[test.label == 1].head(4), test[test.label == 0].head(4)])
parity_inputs = sf.matrix(parity_rows)

In [ ]:
from skl2onnx import to_onnx
from skl2onnx.common.data_types import FloatTensorType

proto = to_onnx(segmenter, initial_types=[("features", FloatTensorType([None, N]))], target_opset=15)
attach(proto, features=sf.FEATURES,
       anomaly_thresholds={str(s): float(t) for s, t in thresholds.items()},
       anomaly_quantile=QUANTILE,
       parity={"inputs": parity_inputs.tolist(), "segment": segmenter.predict(parity_inputs).tolist()})
session = save(proto, "segments_kmeans")

sample = sf.matrix(test.sample(100_000, random_state=1))
outputs = dict(zip([o.name for o in session.get_outputs()], session.run(None, {"features": sample})))
print("outputs:", list(outputs))
print("segments agree:", (outputs["label"].ravel() == segmenter.predict(sample)).mean())